In [1]:
import pandas as pd
from deep_translator import GoogleTranslator

from pathlib import Path
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import joblib

import pandas as pd
import re

START_YEAR = 2019

In [2]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

In [3]:
class Classifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [4]:
def embed(texts, batch_size=32):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, return_tensors='pt').to(device)
            output = model(**encoded)
            cls_embeddings = output.last_hidden_state[:, 0, :]  # [CLS] token
            embeddings.append(cls_embeddings.cpu())
    return torch.cat(embeddings)

CIHR

In [5]:
cihr_path = "raw_data/CIHR/"
cihr_files = Path(cihr_path).glob("*.csv")

CIHR_DFS = [pd.read_csv(f) for f in cihr_files]
CIHR_DATA = pd.concat(CIHR_DFS, ignore_index=True)

In [6]:
grant_descriptors = [
    "ApplicationTitle_TitreDemande", "PrimaryThemeEN_ThemePrincipalAN", "AllResearchCategoriesEN_TousCategoriesRechercheAN", "ApplicationKeywords_MotsClesDemande"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

CIHR_DATA = CIHR_DATA[grant_descriptors]
CIHR_DATA.columns = col_names

CIHR_DATA.drop_duplicates(inplace=True)
CIHR_DATA["Main_Discipline"].value_counts()

Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Not applicable/Specified                            127
Name: count, dtype: int64

Training CIHR Main Discipline

In [7]:
tmp_data = CIHR_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not applicable/Specified"]

tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data["Main_Discipline"].value_counts()


Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Name: count, dtype: int64

In [8]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [9]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 249.2795
Epoch 2: Loss = 180.7364
Epoch 3: Loss = 164.3359
Epoch 4: Loss = 158.2620
Epoch 5: Loss = 155.2451
Epoch 6: Loss = 153.0930
Epoch 7: Loss = 151.3354
Epoch 8: Loss = 149.6686
Epoch 9: Loss = 148.8204
Epoch 10: Loss = 146.9528
Epoch 11: Loss = 146.0776
Epoch 12: Loss = 144.9589
Epoch 13: Loss = 144.0851
Epoch 14: Loss = 143.2253
Epoch 15: Loss = 142.2184
Epoch 16: Loss = 141.6156
Epoch 17: Loss = 140.6598
Epoch 18: Loss = 139.7943
Epoch 19: Loss = 139.6441
Epoch 20: Loss = 138.6479
Epoch 21: Loss = 137.8124
Epoch 22: Loss = 137.1659
Epoch 23: Loss = 135.8403
Epoch 24: Loss = 135.3661
Epoch 25: Loss = 134.7382
Epoch 26: Loss = 134.1570
Epoch 27: Loss = 133.2803
Epoch 28: Loss = 133.2057
Epoch 29: Loss = 132.1437
Epoch 30: Loss = 131.6195
Epoch 31: Loss = 130.4253
Epoch 32: Loss = 130.0358
Epoch 33: Loss = 129.7320
Epoch 34: Loss = 129.1782
Epoch 35: Loss = 127.7823
Epoch 36: Loss = 127.3523
Epoch 37: Loss = 126.9187
Epoch 38: Loss = 126.2544
Epoch 39: Loss = 125.

In [10]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                 precision    recall  f1-score   support

                                     Biomedical       0.88      0.90      0.89      1798
                                       Clinical       0.62      0.57      0.60       774
                        Health systems/services       0.64      0.64      0.64       566
Social/Cultural/Environmental/Population Health       0.68      0.68      0.68       601

                                       accuracy                           0.76      3739
                                      macro avg       0.71      0.70      0.70      3739
                                   weighted avg       0.76      0.76      0.76      3739



In [11]:
torch.save(clf_model.state_dict(), "models/CIHR_MD.pt")
joblib.dump(label_mapping, "models/CIHR_MD_label_mapping.pkl")


['models/CIHR_MD_label_mapping.pkl']

NSERC

In [12]:
nserc_path = "raw_data/NSERC/"
nserc_files = Path(nserc_path).glob("*.csv")

NSERC_DFS = [pd.read_csv(f) for f in nserc_files]
NSERC_DATA = pd.concat(NSERC_DFS, ignore_index=True)

In [13]:
grant_descriptors = [
    "ApplicationTitle", "AreaOfApplicationGroupEN", "ResearchSubjectEN", "Keyword"
]

col_names = [
     'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

NSERC_DATA = NSERC_DATA[grant_descriptors]
NSERC_DATA.columns = col_names

NSERC_DATA.drop_duplicates(inplace=True)

Model for NSERC Main Discipline

In [14]:
tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

tmp_data = tmp_data[~(tmp_data["Main_Discipline"].isin(["Not available", "Advancement of knowledge"]))]

tmp_data["Main_Discipline"].value_counts()

Main_Discipline
Manufacturing processes and products       4626
Information and communication services     3936
Environment                                3927
Energy resources                           3123
Health, education and social services      2937
Transportation systems and services        2077
Agriculture and primary food production    1541
Construction, urban and rural planning     1376
Northern development                       1203
Natural resources (economic aspects)       1106
The socioeconomic objective available       737
Commercial services                         400
Name: count, dtype: int64

In [15]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.tolist())
X_test = embed(X_test_text.tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [16]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 692.9776
Epoch 2: Loss = 509.2633
Epoch 3: Loss = 463.0170
Epoch 4: Loss = 444.8086
Epoch 5: Loss = 435.7936
Epoch 6: Loss = 428.1474
Epoch 7: Loss = 421.3403
Epoch 8: Loss = 415.9364
Epoch 9: Loss = 411.8096
Epoch 10: Loss = 406.8668
Epoch 11: Loss = 403.7502
Epoch 12: Loss = 400.7308
Epoch 13: Loss = 397.7212
Epoch 14: Loss = 393.9669
Epoch 15: Loss = 391.1496
Epoch 16: Loss = 387.4155
Epoch 17: Loss = 385.7679
Epoch 18: Loss = 382.6929
Epoch 19: Loss = 379.7475
Epoch 20: Loss = 377.0181
Epoch 21: Loss = 374.3423
Epoch 22: Loss = 371.8591
Epoch 23: Loss = 369.5487
Epoch 24: Loss = 368.0466
Epoch 25: Loss = 364.2503
Epoch 26: Loss = 363.1859
Epoch 27: Loss = 360.1204
Epoch 28: Loss = 358.2304
Epoch 29: Loss = 354.9563
Epoch 30: Loss = 353.0154
Epoch 31: Loss = 351.3669
Epoch 32: Loss = 350.1488
Epoch 33: Loss = 346.4176
Epoch 34: Loss = 345.7488
Epoch 35: Loss = 343.2401
Epoch 36: Loss = 340.7032
Epoch 37: Loss = 338.5095
Epoch 38: Loss = 336.6220
Epoch 39: Loss = 335.

In [17]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                         precision    recall  f1-score   support

Agriculture and primary food production       0.81      0.88      0.84       308
                    Commercial services       0.68      0.53      0.59        80
 Construction, urban and rural planning       0.83      0.73      0.78       275
                       Energy resources       0.81      0.82      0.81       625
                            Environment       0.78      0.85      0.81       786
  Health, education and social services       0.81      0.82      0.81       588
 Information and communication services       0.86      0.88      0.87       787
   Manufacturing processes and products       0.72      0.76      0.74       925
   Natural resources (economic aspects)       0.79      0.78      0.79       221
                   Northern development       0.77      0.65      0.70       241
  The socioeconomic objective available       0.37      0.14      0.21       147
    Transportation systems 

In [18]:
torch.save(clf_model.state_dict(), "models/NSERC_MD.pt")
joblib.dump(label_mapping, "models/NSERC_MD_label_mapping.pkl")

['models/NSERC_MD_label_mapping.pkl']

Model for NSERC Area of Research

In [19]:
tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle
translator = GoogleTranslator(target="en")
                              
val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 50].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

for class_name in classes:
    tmp_data["Area_of_Research"] = tmp_data["Area_of_Research"].replace(class_name, translator.translate(class_name))

val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 100].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].str.replace(r' \(.*?\)', '', regex=True)
                              
tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
tmp_data = tmp_data[~(tmp_data["Area_of_Research"].isin(["Not available", "The field of research available"]))]

tmp_data = tmp_data.dropna(subset=['Area_of_Research'])
tmp_data["Area_of_Research"].value_counts()

Area_of_Research
Information technology              500
Materials science and technology    500
Inorganic chemistry                 500
Chemical engineering                500
Molecular biology                   500
                                   ... 
Animal nutrition and husbandry      105
Photonics                           104
Fuel and energy technology          102
Enzymes                             102
Database management                 101
Name: count, Length: 135, dtype: int64

In [20]:
label_mapping = dict(enumerate(tmp_data['Area_of_Research'].astype('category').cat.categories))
tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Area_of_Research'], test_size=0.2, stratify=tmp_data['Area_of_Research'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [21]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 2221.2285
Epoch 2: Loss = 1876.6856
Epoch 3: Loss = 1634.5530
Epoch 4: Loss = 1506.0900
Epoch 5: Loss = 1436.0797
Epoch 6: Loss = 1390.2945
Epoch 7: Loss = 1355.0251
Epoch 8: Loss = 1330.0321
Epoch 9: Loss = 1312.2940
Epoch 10: Loss = 1298.0538
Epoch 11: Loss = 1281.4933
Epoch 12: Loss = 1268.5521
Epoch 13: Loss = 1256.2753
Epoch 14: Loss = 1246.9256
Epoch 15: Loss = 1236.6649
Epoch 16: Loss = 1233.1610
Epoch 17: Loss = 1224.8185
Epoch 18: Loss = 1212.5657
Epoch 19: Loss = 1208.1511
Epoch 20: Loss = 1202.2544
Epoch 21: Loss = 1195.8058
Epoch 22: Loss = 1190.3555
Epoch 23: Loss = 1181.3076
Epoch 24: Loss = 1178.3547
Epoch 25: Loss = 1172.8520
Epoch 26: Loss = 1168.6996
Epoch 27: Loss = 1162.3151
Epoch 28: Loss = 1159.3952
Epoch 29: Loss = 1155.9182
Epoch 30: Loss = 1149.0460
Epoch 31: Loss = 1147.2103
Epoch 32: Loss = 1142.2211
Epoch 33: Loss = 1138.7542
Epoch 34: Loss = 1132.0992
Epoch 35: Loss = 1129.4442
Epoch 36: Loss = 1126.6777
Epoch 37: Loss = 1121.8182
Epoch 38: 

In [22]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                            precision    recall  f1-score   support

                                    Advanced manufacturing       0.32      0.34      0.33        32
        Aerospace, aeronautical and automotive engineering       0.34      0.43      0.38        92
                                  Agricultural engineering       0.43      0.45      0.44        22
                                                Algorithms       0.38      0.24      0.29        34
                                      Analytical chemistry       0.31      0.35      0.33       100
                                            Animal biology       0.13      0.12      0.13       100
                                            Animal ecology       0.41      0.45      0.43       100
                            Animal nutrition and husbandry       0.73      0.52      0.61        21
                          Animal physiology and metabolism       0.35      0.39      0.37       100

In [23]:
torch.save(clf_model.state_dict(), "models/NSERC_AR.pt")
joblib.dump(label_mapping, "models/NSERC_AR_label_mapping.pkl")

['models/NSERC_AR_label_mapping.pkl']

SSHRC

In [24]:
sshrc_path = "raw_data/SSHRC/"
sshrc_files = Path(sshrc_path).glob("*.csv")

SSHRC_DFS = [pd.read_csv(f) for f in sshrc_files]
SSHRC_DATA = pd.concat(SSHRC_DFS, ignore_index=True)

In [25]:
grant_descriptors = [
    "Title-Titre", "Area_of_Research", "Main_Discipline", "Keywords-Mots-clés"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]


SSHRC_DATA = SSHRC_DATA[grant_descriptors]
SSHRC_DATA.columns = col_names

SSHRC_DATA.drop_duplicates(inplace=True)

Model for SSHRC Main Discipline

In [26]:
tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[~(tmp_data['Main_Discipline'].isin(["Not Specified", "Not Subject to Research Classification"]))]

val_counts = tmp_data["Main_Discipline"].value_counts()
classes = val_counts[val_counts > 100].index
tmp_data = tmp_data[tmp_data["Main_Discipline"].isin(classes)]

tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data["Main_Discipline"].value_counts()

Main_Discipline
Children                                                500
Communication                                           500
Multiculturalism and ethnic studies                     500
Violence                                                500
Information Technologies                                500
Politics and government                                 500
Science and technology                                  500
Post-Secondary Education and Research                   500
Arts and culture                                        500
Women                                                   500
Health                                                  500
Employment and labour                                   500
Gender Issues                                           500
Family                                                  500
Economic and Regional Development                       500
Indigenous peoples                                      500
Social development and w

In [27]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [28]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 811.5587
Epoch 2: Loss = 743.6655
Epoch 3: Loss = 667.6952
Epoch 4: Loss = 609.8155
Epoch 5: Loss = 570.5573
Epoch 6: Loss = 545.0717
Epoch 7: Loss = 526.7764
Epoch 8: Loss = 515.3699
Epoch 9: Loss = 504.5385
Epoch 10: Loss = 497.0589
Epoch 11: Loss = 490.7236
Epoch 12: Loss = 485.1603
Epoch 13: Loss = 480.7515
Epoch 14: Loss = 475.1909
Epoch 15: Loss = 470.9226
Epoch 16: Loss = 468.2179
Epoch 17: Loss = 465.3458
Epoch 18: Loss = 462.8780
Epoch 19: Loss = 459.6280
Epoch 20: Loss = 456.1841
Epoch 21: Loss = 453.4003
Epoch 22: Loss = 450.8619
Epoch 23: Loss = 449.5015
Epoch 24: Loss = 446.7172
Epoch 25: Loss = 445.2391
Epoch 26: Loss = 443.4574
Epoch 27: Loss = 442.2642
Epoch 28: Loss = 440.0003
Epoch 29: Loss = 437.8465
Epoch 30: Loss = 435.4234
Epoch 31: Loss = 433.8680
Epoch 32: Loss = 432.2114
Epoch 33: Loss = 430.3802
Epoch 34: Loss = 428.5974
Epoch 35: Loss = 426.3905
Epoch 36: Loss = 424.4231
Epoch 37: Loss = 422.8902
Epoch 38: Loss = 421.4266
Epoch 39: Loss = 420.

In [29]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                      precision    recall  f1-score   support

                                         Agriculture       0.60      0.70      0.65        50
                                    Arts and culture       0.35      0.40      0.37       100
                         Canada's Official Languages       0.48      0.57      0.52        21
                                            Children       0.54      0.58      0.56       100
                                  Children and youth       0.20      0.10      0.13        30
                                       Communication       0.34      0.37      0.35       100
                   Economic and Regional Development       0.48      0.44      0.46       100
                                           Education       0.45      0.46      0.46       100
                                             Elderly       0.69      0.80      0.74        93
                               Employment and labour       

In [30]:
torch.save(clf_model.state_dict(), "models/SSHRC_MD.pt")
joblib.dump(label_mapping, "models/SSHRC_MD_label_mapping.pkl")

['models/SSHRC_MD_label_mapping.pkl']

Model for SSHRC Area of Research

In [31]:
SSHRC_DATA["Area_of_Research"].value_counts()

tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 200].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
tmp_data = tmp_data.dropna(subset=['Area_of_Research'])
tmp_data = tmp_data[~(tmp_data['Area_of_Research'].isin(["Not Specified", "Not specified", "Not Applicable", "Multiple primary fields of research", "Interdisciplinary Studies"]))]

tmp_data["Area_of_Research"].value_counts()

Area_of_Research
Linguistics                                          500
History                                              500
Criminology                                          500
Economics                                            500
Political Science                                    500
Philosophy                                           500
Social Work                                          500
Geography                                            500
Management, Business, Administrative Studies         500
Fine Arts                                            500
Urban and Regional Studies, Environmental Studies    500
Communications and Media Studies                     500
Psychology                                           500
Anthropology                                         500
Law                                                  500
Sociology                                            500
Education                                            500
Literature, Mo

In [32]:
label_mapping = dict(enumerate(tmp_data['Area_of_Research'].astype('category').cat.categories))
tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Area_of_Research'], test_size=0.2, stratify=tmp_data['Area_of_Research'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [33]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 413.0439
Epoch 2: Loss = 392.2808
Epoch 3: Loss = 362.7834
Epoch 4: Loss = 332.7492
Epoch 5: Loss = 307.3504
Epoch 6: Loss = 289.2706
Epoch 7: Loss = 275.5120
Epoch 8: Loss = 265.4506
Epoch 9: Loss = 257.9167
Epoch 10: Loss = 251.7357
Epoch 11: Loss = 246.9155
Epoch 12: Loss = 242.3710
Epoch 13: Loss = 238.7789
Epoch 14: Loss = 236.2051
Epoch 15: Loss = 233.3330
Epoch 16: Loss = 230.8725
Epoch 17: Loss = 228.0825
Epoch 18: Loss = 225.9603
Epoch 19: Loss = 224.6891
Epoch 20: Loss = 222.3646
Epoch 21: Loss = 220.5323
Epoch 22: Loss = 219.0016
Epoch 23: Loss = 217.5087
Epoch 24: Loss = 216.5022
Epoch 25: Loss = 215.1267
Epoch 26: Loss = 213.5490
Epoch 27: Loss = 212.4336
Epoch 28: Loss = 211.8322
Epoch 29: Loss = 209.7974
Epoch 30: Loss = 208.4904
Epoch 31: Loss = 207.2898
Epoch 32: Loss = 206.3571
Epoch 33: Loss = 206.1425
Epoch 34: Loss = 204.6116
Epoch 35: Loss = 203.3699
Epoch 36: Loss = 202.9466
Epoch 37: Loss = 200.8538
Epoch 38: Loss = 200.5590
Epoch 39: Loss = 200.

In [34]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                   precision    recall  f1-score   support

                                     Anthropology       0.26      0.28      0.27       100
                                      Archaeology       0.82      0.82      0.82       100
             Classics, Classical & Dead Languages       0.77      0.69      0.73        54
                 Communications and Media Studies       0.46      0.44      0.45       100
                                      Criminology       0.61      0.70      0.65       100
                                       Demography       0.51      0.54      0.52        52
                                        Economics       0.64      0.58      0.61       100
                                        Education       0.51      0.43      0.46       100
                                        Fine Arts       0.43      0.48      0.45       100
                                        Geography       0.39      0.34      0.36       10

In [35]:
torch.save(clf_model.state_dict(), "models/SSHRC_AR.pt")
joblib.dump(label_mapping, "models/SSHRC_AR_label_mapping.pkl")

['models/SSHRC_AR_label_mapping.pkl']